# Unsupervised Anomaly Detection

## Objective
Evaluate three unsupervised anomaly detection methods that require no labeled training data:
1. **Isolation Forest** — Tree-based isolation of anomalies via random partitioning
2. **Local Outlier Factor (LOF)** — Density-based detection using local neighborhood deviation
3. **One-Class SVM** — Kernel-based boundary learning around normal data

These models are trained **only on normal data** and evaluated on the full test set.

In [ ]:
import sys, os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath('..'))

from src.models import train_unsupervised_models
from src.evaluation import (
    evaluate_unsupervised, plot_confusion_matrices,
    plot_roc_pr_curves, print_classification_report,
)
from src.data_loader import split_data

DATA_DIR = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results', 'figures')
print('Libraries loaded')

## 4.1 Load and Prepare Data

In [ ]:
plant_df = pd.read_csv(os.path.join(DATA_DIR, 'plant_engineered.csv'))

with open(os.path.join(DATA_DIR, 'engineered_feature_names.json'), 'r') as f:
    feat_names = json.load(f)

feature_cols = feat_names['all']
feature_cols = [c for c in feature_cols if c in plant_df.columns]

# Drop NaN rows from rolling features
plant_clean = plant_df.dropna(subset=feature_cols).reset_index(drop=True)

# Split
X_train, X_test, y_train, y_test = split_data(plant_clean, feature_cols)

# Scale
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

# Extract normal-only training data for unsupervised methods
X_train_normal = X_train_scaled[y_train == 0]
print(f'\nNormal training samples: {len(X_train_normal):,}')
print(f'Test set: {len(X_test):,} (anomaly rate: {y_test.mean():.3f})')

## 4.2 Train Unsupervised Models

In [ ]:
unsup_results = train_unsupervised_models(X_train_normal, contamination=0.08)

## 4.3 Evaluation

In [ ]:
unsup_eval = evaluate_unsupervised(unsup_results, X_test_scaled, y_test)
print('\nUnsupervised Model Results:')
print(unsup_eval[['precision', 'recall', 'f1_score', 'mcc', 'roc_auc', 'train_time']].round(4).to_string())

In [ ]:
# Confusion matrices
plot_confusion_matrices(unsup_results, X_test_scaled, y_test,
                        model_type='unsupervised',
                        save_path=os.path.join(RESULTS_DIR, 'unsupervised_confusion.png'))

In [ ]:
# ROC and PR curves
plot_roc_pr_curves(unsup_results, X_test_scaled, y_test,
                   model_type='unsupervised',
                   save_path=os.path.join(RESULTS_DIR, 'unsupervised_roc_pr.png'))

In [ ]:
# Detailed report for best model
best_name = unsup_eval['f1_score'].idxmax()
y_pred_raw = unsup_results[best_name]['model'].predict(X_test_scaled)
y_pred = np.where(y_pred_raw == -1, 1, 0)
print_classification_report(y_test, y_pred, title=f'Best Unsupervised: {best_name}')

## 4.4 Save Results

In [ ]:
unsup_eval.to_csv(os.path.join(DATA_DIR, 'unsupervised_results.csv'))
print('Saved unsupervised results')

## Summary

Unsupervised methods provide a baseline without requiring labeled anomalies. Isolation Forest typically performs best due to its efficiency with high-dimensional data.

